# 05b — Feature Engineering V2: Rolling Rates + Turnaround Buffer

Adds flight number delay rates, origin-hour delay rates, carrier-origin delay rates, hourly congestion, and scheduled turnaround buffer.

**Input:** `dataset/merged_flights_fe.parquet`

**Output:** `dataset/merged_flights_fe.parquet` (updated)

**Features added:** flight_num_delay_rate_30d, origin_hour_delay_rate_30d, carrier_origin_delay_rate_30d, hourly_flight_count, scheduled_turnaround_buffer

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

flights = pd.read_parquet("dataset/merged_flights_fe.parquet")

borderline = [
    'route_delay_rate_30d', 'origin_wind_dir', 'dest_wind_dir',
    'IS_HOLIDAY', 'origin_precip_1hr', 'cascade_score', 'dest_precip_1hr',
    'dest_wind_speed', 'is_summer_peak', 'origin_wind_speed',
    'origin_weather_severity', 'is_fall_low', 'dest_weather_severity',
    'is_regional', 'origin_sky_condition', 'airport_cluster_label',
    'dest_sky_condition', 'is_high_delay_day', 'is_low_delay_day'
]

print(f"\n{'Feature':<30} {'Importance':<12} {'Null %':<10} {'Unique':<10}")
print("=" * 62)
for f in borderline:
    null_pct = flights[f].isna().mean() * 100
    uniq = flights[f].nunique()
    imp_map = {
        'route_delay_rate_30d': 480, 'origin_wind_dir': 422, 'dest_wind_dir': 377,
        'IS_HOLIDAY': 348, 'origin_precip_1hr': 336, 'cascade_score': 324,
        'dest_precip_1hr': 311, 'dest_wind_speed': 291, 'is_summer_peak': 254,
        'origin_wind_speed': 227, 'origin_weather_severity': 217, 'is_fall_low': 216,
        'dest_weather_severity': 194, 'is_regional': 168, 'origin_sky_condition': 162,
        'airport_cluster_label': 143, 'dest_sky_condition': 133,
        'is_high_delay_day': 124, 'is_low_delay_day': 120
    }
    imp = imp_map[f]
    print(f"{f:<30} {imp:<12} {null_pct:<10.1f} {uniq:<10}")


Feature                        Importance   Null %     Unique    
route_delay_rate_30d           480          0.7        388008    
origin_wind_dir                422          12.1       49        
dest_wind_dir                  377          12.5       50        
IS_HOLIDAY                     348          0.0        2         
origin_precip_1hr              336          11.0       261       
cascade_score                  324          0.0        7         
dest_precip_1hr                311          11.0       272       
dest_wind_speed                291          8.7        83        
is_summer_peak                 254          0.0        2         
origin_wind_speed              227          8.6        85        
origin_weather_severity        217          48.4       842       
is_fall_low                    216          0.0        2         
dest_weather_severity          194          49.6       853       
is_regional                    168          0.0        2         
origin_sk

In [3]:
flights.columns

Index(['YEAR', 'QUARTER', 'MONTH', 'DAY_OF_MONTH', 'DAY_OF_WEEK', 'FL_DATE',
       'OP_UNIQUE_CARRIER', 'OP_CARRIER_AIRLINE_ID', 'TAIL_NUM',
       'OP_CARRIER_FL_NUM',
       ...
       'origin_weather_severity', 'dest_weather_severity',
       'airline_cluster_label', 'airport_cluster_label',
       'flight_num_delay_rate_30d_x', 'origin_hour_delay_rate_30d',
       'hourly_flight_count', 'flight_num_delay_rate_30d_y',
       'scheduled_turnaround_buffer', 'carrier_origin_delay_rate_30d'],
      dtype='object', length=105)

In [4]:
pd.set_option("Display.Max_rows",None)
pd.set_option("Display.Max_columns",None)
flights.head()

,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,OP_CARRIER_AIRLINE_ID,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_ABR,DEST_STATE_NM,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,TAXI_OUT,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY,AIRLINE_NAME,DEP_HOUR,DAY_NAME,DIST_GROUP,TAXI_GROUP,IS_HOLIDAY,ARR_HOUR,origin_temp,origin_dew_point,origin_pressure,origin_wind_dir,origin_wind_speed,origin_sky_condition,origin_precip_1hr,dest_temp,dest_dew_point,dest_pressure,dest_wind_dir,dest_wind_speed,dest_sky_condition,dest_precip_1hr,OP_REVENUES,OP_EXPENSES,NET_INCOME,OP_PROFIT_LOSS,FLYING_OPS,MAINTENANCE,PAX_SERVICE,TRANS_REV_PAX,DEPREC_AMORT,TRANS_EXPENSES,origin_monthly_passengers,origin_monthly_departures,is_early_departure,is_peak_hours,is_high_delay_day,is_low_delay_day,is_summer_peak,is_fall_low,profit_margin,maintenance_ratio,revenue_per_pax,cost_efficiency,is_regional,is_profitable,origin_is_adverse_sky,dest_is_adverse_sky,DEP_DATETIME,cascade_score,cascade_delay_minutes,hours_since_last_delay,airline_delay_rate_30d,origin_delay_rate_30d,dest_delay_rate_30d,route_delay_rate_30d,origin_avg_taxi_out_30d,origin_weather_severity,dest_weather_severity,airline_cluster_label,airport_cluster_label,flight_num_delay_rate_30d_x,origin_hour_delay_rate_30d,hourly_flight_count,flight_num_delay_rate_30d_y,scheduled_turnaround_buffer,carrier_origin_delay_rate_30d
0,2023,1,1,1,7,2023-01-01,9E,20363,N131EV,5244.0,ORD,"Chicago, IL",IL,Illinois,JFK,"New York, NY",NY,New York,1520,1524.0,4.0,4.0,0.0,20.0,23.0,1841,1838.0,-3.0,0.0,0.0,141.0,134.0,91.0,740.0,NaN,NaN,NaN,NaN,NaN,Endeavor Air,15,Sun,500-1K,10-20,1,18,5.6,3.9,1012.8,260.0,2.6,NaN,0.0,12.8,3.9,1013.7,290.0,6.7,4.0,0.0,143826.04,233156.8,-89175.51,-89330.76,116882.94,69991.6,25028.93,143826.04,3109.96,0.0,1855892.0,416.0,0,1,1,0,0,0,-0.621103,0.300191,1.0,1.621103,1,0,NaN,0.0,2023-01-01 15:20:00,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.222667,Reliable Regionals,Major Hubs,NaN,NaN,24,NaN,216.0,NaN
1,2023,1,1,1,7,2023-01-01,9E,20363,N131EV,5317.0,JFK,"New York, NY",NY,New York,ORD,"Chicago, IL",IL,Illinois,945,941.0,-4.0,0.0,0.0,25.0,8.0,1144,1120.0,-24.0,0.0,0.0,179.0,159.0,126.0,740.0,NaN,NaN,NaN,NaN,NaN,Endeavor Air,9,Sun,500-1K,20-30,1,11,8.9,8.9,1008.6,250.0,5.1,6.0,0.0,3.3,2.2,1010.3,140.0,2.1,NaN,-0.1,143826.04,233156.8,-89175.51,-89330.76,116882.94,69991.6,25028.93,143826.04,3109.96,0.0,1075492.0,160.0,1,0,1,0,0,0,-0.621103,0.300191,1.0,1.621103,1,0,1.0,NaN,2023-01-01 09:45:00,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.268000,NaN,Reliable Regionals,Major Hubs,NaN,NaN,25,NaN,NaN,NaN
2,2023,1,1,1,7,2023-01-01,9E,20363,N131EV,5397.0,JFK,"New York, NY",NY,New York,BGR,"Bangor, ME",ME,Maine,2100,2056.0,-4.0,0.0,0.0,28.0,5.0,2236,2229.0,-7.0,0.0,0.0,96.0,93.0,60.0,382.0,NaN,NaN,NaN,NaN,NaN,Endeavor Air,21,Sun,0-500,20-30,1,22,12.2,0.6,1014.2,300.0,4.6,4.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,143826.04,233156.8,-89175.51,-89330.76,116882.94,69991.6,25028.93,143826.04,3109.96,0.0,1075492.0,160.0,0,1,1,0,0,0,-0.621103,0.300191,1.0,1.621103,1,0,0.0,NaN,2023-01-01 21:00:00,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.194667,NaN,Reliable Regionals,Major Hubs,NaN,NaN,12,NaN,139.0,NaN
3,2023,1,1,1,7,2023-01-01,9E,20363,N133EV,5076.0,ATL,"Atlanta, GA",GA,Georgia,SGF,"Springfield, MO",MO,Missouri,1130,1125.0,-5.0,0.0,0.0,17.0,4.0,1225,1214.0,-11.0,0.0,0.0,115.0,109.0,88.0,563.0,NaN,NaN,NaN,NaN,NaN,Endeavor Air,11,Sun,500-1K,10-20,1,12,12.2,11.7,1017.8,0.0,0.0,9.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,143826.04,233156.8,-89175.51,-89330.76,116882.94,69991.6,25028.93,143826.04,3109.96,0.0,3194785.0,358.0,0,0,1,0,0,0,-0.621103,0.300191,1.0,1.621103,1,0,1.0,NaN,2023-01-01 11:30:00,0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.300000,NaN,Reliable Regionals,Major Hubs,NaN,NaN,57,NaN,185.0,NaN
4,2023,1,1,1,7,2023-0

In [5]:
print("Delay rate by QUARTER:")
print(flights.groupby('QUARTER')['ARR_DEL15'].mean().round(4) * 100)

print("\nDelay rate by MONTH:")
print(flights.groupby('MONTH')['ARR_DEL15'].mean().round(4) * 100)

print("\nVariance within each quarter:")
for q in [1, 2, 3, 4]:
    months = flights[flights['QUARTER'] == q].groupby('MONTH')['ARR_DEL15'].mean() * 100
    print(f"  Q{q}: {months.round(1).to_dict()} — spread: {months.max() - months.min():.1f} pts")

Delay rate by QUARTER:
QUARTER
1    20.57
2    23.37
3    23.78
4    15.65
Name: ARR_DEL15, dtype: float64

Delay rate by MONTH:
MONTH
1     21.67
2     18.49
3     21.39
4     20.50
5     22.76
6     26.74
7     29.03
8     22.70
9     17.06
10    14.39
11    14.13
12    18.48
Name: ARR_DEL15, dtype: float64

Variance within each quarter:
  Q1: {1: 21.7, 2: 18.5, 3: 21.4} — spread: 3.2 pts
  Q2: {4: 20.5, 5: 22.8, 6: 26.7} — spread: 6.2 pts
  Q3: {7: 29.0, 8: 22.7, 9: 17.1} — spread: 12.0 pts
  Q4: {10: 14.4, 11: 14.1, 12: 18.5} — spread: 4.4 pts


In [6]:
print("Delay rate by MONTH:")
print(flights.groupby('MONTH')['ARR_DEL15'].mean().round(4) * 100)

print("\nMonths NOT captured by any flag:")
print("  is_summer_peak covers: 6, 7")
print("  is_fall_low covers: 10, 11")
print("  Uncovered: 1, 2, 3, 4, 5, 8, 9, 12")

uncovered = flights[~flights['MONTH'].isin([6, 7, 10, 11])]
print(f"\n  Uncovered months delay rate range:")
print(uncovered.groupby('MONTH')['ARR_DEL15'].mean().round(4) * 100)

Delay rate by MONTH:
MONTH
1     21.67
2     18.49
3     21.39
4     20.50
5     22.76
6     26.74
7     29.03
8     22.70
9     17.06
10    14.39
11    14.13
12    18.48
Name: ARR_DEL15, dtype: float64

Months NOT captured by any flag:
  is_summer_peak covers: 6, 7
  is_fall_low covers: 10, 11
  Uncovered: 1, 2, 3, 4, 5, 8, 9, 12

  Uncovered months delay rate range:
MONTH
1     21.67
2     18.49
3     21.39
4     20.50
5     22.76
8     22.70
9     17.06
12    18.48
Name: ARR_DEL15, dtype: float64


In [ ]:
carrier_daily_counts = flights.groupby(['OP_UNIQUE_CARRIER', 'MONTH']).apply(
    lambda x: x.groupby('FL_DATE').ngroups
).reset_index(name='days_with_flights')

print("Average days with flights per month per carrier:")
print(carrier_daily_counts.groupby('MONTH')['days_with_flights'].mean().round(1))

aa_daily = flights[flights['OP_UNIQUE_CARRIER'] == 'AA'].groupby('FL_DATE')['ARR_DEL15'].mean().reset_index()
aa_daily = aa_daily.sort_values('FL_DATE')
aa_daily['rolling_30'] = aa_daily['ARR_DEL15'].shift(1).rolling(30, min_periods=7).mean()

print("\nAA around Feb 2024:")
mask = (aa_daily['FL_DATE'] >= '2024-01-25') & (aa_daily['FL_DATE'] <= '2024-02-20')
print(aa_daily[mask][['FL_DATE', 'ARR_DEL15', 'rolling_30']].to_string(index=False))

Average days with flights per month per carrier:
MONTH
1     90.9
2     83.1
3     90.9
4     88.0
5     90.9
6     88.0
7     90.9
8     87.2
9     60.0
10    62.0
11    60.0
12    62.0
Name: days_with_flights, dtype: float64

AA around Feb 2024:
   FL_DATE  ARR_DEL15  rolling_30
2024-01-25   0.249906    0.311061
2024-01-26   0.254998    0.309466
2024-01-27   0.197792    0.309151
2024-01-28   0.230886    0.305684
2024-01-29   0.130137    0.306034
2024-01-30   0.139565    0.303695
2024-01-31   0.118470    0.303182
2024-02-01   0.169932    0.302388
2024-02-02   0.186511    0.300380
2024-02-03   0.149603    0.299188
2024-02-04   0.297637    0.299149
2024-02-05   0.230975    0.302977
2024-02-06   0.156264    0.298258
2024-02-07   0.122525    0.293750
2024-02-08   0.185871    0.284287
2024-02-09   0.150339    0.270223
2024-02-10   0.193905    0.265578
2024-02-11   0.257869    0.262949
2024-02-12   0.220023    0.253419
2024-02-13   0.167622    0.248812
2024-02-14   0.111111    0.241771
2024

In [8]:
print("Delay rate by DAY_OF_MONTH:")
print(flights.groupby('DAY_OF_MONTH')['ARR_DEL15'].mean().round(4) * 100)

print(f"\nSpread: {flights.groupby('DAY_OF_MONTH')['ARR_DEL15'].mean().max()*100:.1f}% - {flights.groupby('DAY_OF_MONTH')['ARR_DEL15'].mean().min()*100:.1f}% = {(flights.groupby('DAY_OF_MONTH')['ARR_DEL15'].mean().max() - flights.groupby('DAY_OF_MONTH')['ARR_DEL15'].mean().min())*100:.1f} pts")

Delay rate by DAY_OF_MONTH:
DAY_OF_MONTH
1     20.35
2     20.66
3     20.74
4     19.77
5     21.25
6     21.40
7     20.55
8     20.18
9     21.17
10    21.47
11    21.74
12    21.35
13    20.42
14    21.63
15    22.72
16    23.70
17    22.66
18    21.74
19    21.43
20    21.39
21    20.60
22    21.58
23    20.55
24    20.72
25    20.89
26    22.17
27    21.40
28    20.85
29    20.94
30    21.49
31    21.36
Name: ARR_DEL15, dtype: float64

Spread: 23.7% - 19.8% = 3.9 pts


In [9]:
print(f"Unique flight numbers: {flights['OP_CARRIER_FL_NUM'].nunique()}")
print(f"Unique carrier+flight combos: {flights.groupby(['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM']).ngroups}")
print(f"\nFlights per flight number:")
print(flights.groupby('OP_CARRIER_FL_NUM').size().describe())

Unique flight numbers: 6933
Unique carrier+flight combos: 36792

Flights per flight number:
count    6933.000000
mean     2629.135295
std      1663.100747
min         1.000000
25%      1390.000000
50%      2510.000000
75%      3889.000000
max      8243.000000
dtype: float64


In [ ]:
train_temp = flights[flights['FL_DATE'] < '2025-01-01']
flight_num_delay = train_temp.groupby(['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM'])['ARR_DEL15'].mean()

flight_num_map = train_temp.copy()
flight_num_map['flight_num_delay_rate'] = train_temp.set_index(['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM']).index.map(flight_num_delay)

corr = flight_num_map[['flight_num_delay_rate', 'route_delay_rate_30d']].corr()
print("Correlation between flight_num target encoding and route_delay_rate_30d:")
print(corr.round(4))

Correlation between flight_num target encoding and route_delay_rate_30d:
                       flight_num_delay_rate  route_delay_rate_30d
flight_num_delay_rate                 1.0000                0.2876
route_delay_rate_30d                  0.2876                1.0000


## Flight Number Rolling Delay Rate (30d)
Point-in-time safe rolling delay rate per carrier + flight number.

In [ ]:
flight_daily = (
    flights.groupby(['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'FL_DATE'])['ARR_DEL15']
    .mean()
    .reset_index()
    .rename(columns={'ARR_DEL15': 'daily_delay_rate'})
)

flight_daily = flight_daily.sort_values(['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'FL_DATE'])

flight_daily['flight_num_delay_rate_30d'] = (
    flight_daily
    .groupby(['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM'])['daily_delay_rate']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
)

print(f"flight_daily shape: {flight_daily.shape}")
print(f"Nulls: {flight_daily['flight_num_delay_rate_30d'].isna().sum():,}")

flights = flights.merge(
    flight_daily[['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'FL_DATE', 'flight_num_delay_rate_30d']],
    on=['OP_UNIQUE_CARRIER', 'OP_CARRIER_FL_NUM', 'FL_DATE'],
    how='left'
)

print(f"\nShape after merge: {flights.shape}")
print(f"Nulls in flights: {flights['flight_num_delay_rate_30d'].isna().sum():,}")
print(f"\nCorrelation with other rolling features:")
print(flights[['flight_num_delay_rate_30d', 'route_delay_rate_30d', 
               'airline_delay_rate_30d', 'origin_delay_rate_30d']].corr().round(4))

flight_daily shape: (15067950, 5)
Nulls: 248,772

Shape after merge: (18227796, 106)
Nulls in flights: 291,662

Correlation with other rolling features:
                           flight_num_delay_rate_30d  route_delay_rate_30d  \
flight_num_delay_rate_30d                     1.0000                0.5458   
route_delay_rate_30d                          0.5458                1.0000   
airline_delay_rate_30d                        0.4416                0.5659   
origin_delay_rate_30d                         0.4187                0.7073   

                           airline_delay_rate_30d  origin_delay_rate_30d  
flight_num_delay_rate_30d                  0.4416                 0.4187  
route_delay_rate_30d                       0.5659                 0.7073  
airline_delay_rate_30d                     1.0000                 0.6417  
origin_delay_rate_30d                      0.6417                 1.0000  


In [12]:
flights.to_parquet("dataset/merged_flights_fe.parquet", 
                   index=False, engine="pyarrow", compression="snappy")
import os
size_mb = os.path.getsize("dataset/merged_flights_fe.parquet") / 1024 / 1024
print(f"Saved. Size: {size_mb:.1f} MB | Shape: {flights.shape}")

Saved. Size: 1173.8 MB | Shape: (18227796, 106)


In [ ]:
print("origin_avg_taxi_out_30d is grouped by ORIGIN + DEP_HOUR: yes")
print("origin_delay_rate_30d is grouped by ORIGIN only: yes (no hour)")

print("\nDelay rate by ORIGIN + DEP_HOUR (sample for ATL):")
atl = flights[flights['ORIGIN'] == 'ATL'].groupby('DEP_HOUR')['ARR_DEL15'].mean() * 100
print(atl.round(1))

origin_avg_taxi_out_30d is grouped by ORIGIN + DEP_HOUR: yes
origin_delay_rate_30d is grouped by ORIGIN only: yes (no hour)

Delay rate by ORIGIN + DEP_HOUR (sample for ATL):
DEP_HOUR
0     66.7
5     11.5
6     11.4
7     13.1
8     12.7
9     15.4
10    16.1
11    16.2
12    17.2
13    18.4
14    20.8
15    23.3
16    26.4
17    27.2
18    28.6
19    29.2
20    29.8
21    26.5
22    24.8
23    31.2
Name: ARR_DEL15, dtype: float64


## Origin-Hour Rolling Delay Rate (30d)
Captures time-of-day delay patterns per airport.

In [ ]:
origin_hour_daily = (
    flights.groupby(['ORIGIN', 'DEP_HOUR', 'FL_DATE'])['ARR_DEL15']
    .mean()
    .reset_index()
    .rename(columns={'ARR_DEL15': 'daily_delay_rate'})
)
origin_hour_daily = origin_hour_daily.sort_values(
    ['ORIGIN', 'DEP_HOUR', 'FL_DATE']).reset_index(drop=True)

def rolling_lag(x):
    return x.shift(1).rolling(window=30, min_periods=7).mean()

origin_hour_daily['origin_hour_delay_rate_30d'] = (
    origin_hour_daily
    .groupby(['ORIGIN', 'DEP_HOUR'])['daily_delay_rate']
    .transform(rolling_lag)
)

print(f"Shape: {origin_hour_daily.shape}")
print(f"Nulls: {origin_hour_daily['origin_hour_delay_rate_30d'].isna().sum():,}")

origin_hour_daily['FL_DATE'] = pd.to_datetime(origin_hour_daily['FL_DATE'])
flights['FL_DATE'] = pd.to_datetime(flights['FL_DATE'])

flights = flights.merge(
    origin_hour_daily[['ORIGIN', 'DEP_HOUR', 'FL_DATE', 'origin_hour_delay_rate_30d']],
    on=['ORIGIN', 'DEP_HOUR', 'FL_DATE'],
    how='left'
)
print(f"Shape after merge: {flights.shape}")
print(f"Nulls: {flights['origin_hour_delay_rate_30d'].isna().sum():,}")

Shape: (2700227, 5)
Nulls: 34,333
Shape after merge: (18227796, 118)
Nulls: 132,093


In [15]:
origin_hour_daily = (
    flights.groupby(['ORIGIN', 'DEP_HOUR', 'FL_DATE'])['ARR_DEL15']
    .mean()
    .reset_index()
    .rename(columns={'ARR_DEL15': 'daily_delay_rate'})
)

origin_hour_daily = origin_hour_daily.sort_values(['ORIGIN', 'DEP_HOUR', 'FL_DATE'])

origin_hour_daily['origin_hour_delay_rate_30d'] = (
    origin_hour_daily
    .groupby(['ORIGIN', 'DEP_HOUR'])['daily_delay_rate']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
)

print(f"Shape: {origin_hour_daily.shape}")
print(f"Nulls: {origin_hour_daily['origin_hour_delay_rate_30d'].isna().sum():,}")

flights = flights.merge(
    origin_hour_daily[['ORIGIN', 'DEP_HOUR', 'FL_DATE', 'origin_hour_delay_rate_30d']],
    on=['ORIGIN', 'DEP_HOUR', 'FL_DATE'],
    how='left'
)

print(f"\nShape after merge: {flights.shape}")
print(f"Nulls in flights: {flights['origin_hour_delay_rate_30d'].isna().sum():,}")
print(f"Null %: {flights['origin_hour_delay_rate_30d'].isna().mean()*100:.2f}%")

Shape: (2700227, 5)
Nulls: 34,333

Shape after merge: (18227796, 108)
Nulls in flights: 132,093
Null %: 0.72%


In [16]:
print([c for c in flights.columns if 'origin_hour' in c])

['origin_hour_delay_rate_30d_x', 'origin_hour_delay_rate_30d_y', 'origin_hour_delay_rate_30d']


In [ ]:
flights.drop(columns=['origin_hour_delay_rate_30d_x', 'origin_hour_delay_rate_30d_y'], inplace=True)

flights = flights.merge(
    origin_hour_daily[['ORIGIN', 'DEP_HOUR', 'FL_DATE', 'origin_hour_delay_rate_30d']],
    on=['ORIGIN', 'DEP_HOUR', 'FL_DATE'],
    how='left'
)

print(f"Shape: {flights.shape}")
print(f"Nulls: {flights['origin_hour_delay_rate_30d'].isna().sum():,}")
print(f"Null %: {flights['origin_hour_delay_rate_30d'].isna().mean()*100:.2f}%")

print(f"\nSample — ATL by hour:")
print(flights[flights['ORIGIN']=='ATL'].groupby('DEP_HOUR')['origin_hour_delay_rate_30d'].mean().round(4) * 100)

Shape: (18227796, 106)
Nulls: 132,093
Null %: 0.72%

Sample — ATL by hour:
DEP_HOUR
0     73.21
5     11.49
6     11.45
7     13.38
8     12.96
9     15.62
10    16.47
11    16.70
12    17.52
13    18.62
14    20.95
15    23.39
16    26.40
17    27.00
18    29.03
19    29.26
20    29.39
21    26.02
22    24.65
23    30.48
Name: origin_hour_delay_rate_30d, dtype: float64


In [19]:
flights.to_parquet("dataset/merged_flights_fe.parquet", 
                   index=False, engine="pyarrow", compression="snappy")
import os
size_mb = os.path.getsize("dataset/merged_flights_fe.parquet") / 1024 / 1024
print(f"Saved. Size: {size_mb:.1f} MB | Shape: {flights.shape}")

Saved. Size: 1173.8 MB | Shape: (18227796, 106)


In [20]:
print(f"Shape: {flights.shape}")
print(f"\n=== Check for duplicate columns ===")
dupes = flights.columns[flights.columns.duplicated()].tolist()
print(f"Duplicate column names: {dupes if dupes else 'None'}")

print(f"\n=== Check for _x _y columns ===")
xy_cols = [c for c in flights.columns if c.endswith('_x') or c.endswith('_y')]
print(f"Suffix columns: {xy_cols if xy_cols else 'None'}")

print(f"\n=== New features added today ===")
new_features = ['flight_num_delay_rate_30d', 'origin_hour_delay_rate_30d']
for f in new_features:
    if f in flights.columns:
        nulls = flights[f].isna().sum()
        null_pct = flights[f].isna().mean() * 100
        print(f"  {f}: nulls={nulls:,} ({null_pct:.2f}%), dtype={flights[f].dtype}")
    else:
        print(f"  {f}: MISSING!")

print(f"\n=== Cluster labels intact ===")
print(f"  airline_cluster_label nulls: {flights['airline_cluster_label'].isna().sum()}")
print(f"  airport_cluster_label nulls: {flights['airport_cluster_label'].isna().sum()}")

print(f"\n=== Row count ===")
print(f"  Expected: 18,227,796")
print(f"  Actual:   {len(flights):,}")

print(f"\n=== Target intact ===")
print(flights['ARR_DEL15'].value_counts())

print(f"\n=== Full column list ({flights.shape[1]} columns) ===")
for i, col in enumerate(flights.columns):
    print(f"  {i+1:3d}. {col}")

Shape: (18227796, 106)

=== Check for duplicate columns ===
Duplicate column names: None

=== Check for _x _y columns ===
Suffix columns: ['flight_num_delay_rate_30d_x', 'flight_num_delay_rate_30d_y']

=== New features added today ===
  flight_num_delay_rate_30d: nulls=291,662 (1.60%), dtype=float64
  origin_hour_delay_rate_30d: nulls=132,093 (0.72%), dtype=float64

=== Cluster labels intact ===
  airline_cluster_label nulls: 0
  airport_cluster_label nulls: 0

=== Row count ===
  Expected: 18,227,796
  Actual:   18,227,796

=== Target intact ===
ARR_DEL15
0.0    14353289
1.0     3874507
Name: count, dtype: int64

=== Full column list (106 columns) ===
    1. YEAR
    2. QUARTER
    3. MONTH
    4. DAY_OF_MONTH
    5. DAY_OF_WEEK
    6. FL_DATE
    7. OP_UNIQUE_CARRIER
    8. OP_CARRIER_AIRLINE_ID
    9. TAIL_NUM
   10. OP_CARRIER_FL_NUM
   11. ORIGIN
   12. ORIGIN_CITY_NAME
   13. ORIGIN_STATE_ABR
   14. ORIGIN_STATE_NM
   15. DEST
   16. DEST_CITY_NAME
   17. DEST_STATE_ABR
   18. DE

## Hourly Flight Count (Congestion)
Number of flights departing from same origin in the same hour.

In [5]:
if 'hourly_flight_count' not in flights.columns:
    hourly_congestion = flights.groupby(
        ['ORIGIN', 'FL_DATE', 'DEP_HOUR']).size().reset_index(name='hourly_flight_count')
    flights['FL_DATE'] = pd.to_datetime(flights['FL_DATE'])
    hourly_congestion['FL_DATE'] = pd.to_datetime(hourly_congestion['FL_DATE'])
    flights = flights.merge(hourly_congestion, on=['ORIGIN', 'FL_DATE', 'DEP_HOUR'], how='left')
else:
    print("hourly_flight_count already exists, skipping")

print(f"Shape: {flights.shape}")
print(f"Nulls: {flights['hourly_flight_count'].isna().sum()}")
print(flights['hourly_flight_count'].describe().round(2))

flights.to_parquet("dataset/merged_flights_fe.parquet",
    index=False, engine="pyarrow", compression="snappy")
import os
size_mb = os.path.getsize("dataset/merged_flights_fe.parquet") / 1024 / 1024
print(f"Saved. Size: {size_mb:.1f} MB | Shape: {flights.shape}")

Shape: (18227796, 120)
Nulls: 0
count    18227796.00
mean           23.19
std            19.87
min             1.00
25%             7.00
50%            18.00
75%            33.00
max           105.00
Name: hourly_flight_count, dtype: float64
Saved. Size: 1526.6 MB | Shape: (18227796, 120)


In [ ]:
train = flights[flights['FL_DATE'] < '2025-01-01']
test = flights[flights['FL_DATE'] >= '2025-01-01']

no_cascade_delayed = test[(test['cascade_score'] == 0) & (test['ARR_DEL15'] == 1)]
total_delayed = test[test['ARR_DEL15'] == 1]

print(f"Total delayed flights in test: {len(total_delayed):,}")
print(f"Delayed with cascade_score=0: {len(no_cascade_delayed):,} ({len(no_cascade_delayed)/len(total_delayed)*100:.1f}%)")

print(f"\nDelay causes for cascade_score=0 delayed flights:")
cause_cols = ['CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']
causes = no_cascade_delayed[cause_cols].mean()
total_cause = causes.sum()
for c in cause_cols:
    pct = causes[c] / total_cause * 100
    print(f"  {c}: {causes[c]:.1f} min avg ({pct:.1f}%)")

print(f"\nProfile of unpredictable delayed flights:")
print(f"  Avg DEP_HOUR: {no_cascade_delayed['DEP_HOUR'].mean():.1f}")
print(f"  Avg DISTANCE: {no_cascade_delayed['DISTANCE'].mean():.0f}")
print(f"  Avg origin_weather_severity: {no_cascade_delayed['origin_weather_severity'].mean():.4f}")
print(f"  Avg airline_delay_rate_30d: {no_cascade_delayed['airline_delay_rate_30d'].mean():.4f}")

Total delayed flights in test: 1,037,836
Delayed with cascade_score=0: 602,377 (58.0%)

Delay causes for cascade_score=0 delayed flights:
  CARRIER_DELAY: 29.5 min avg (42.9%)
  WEATHER_DELAY: 5.4 min avg (7.9%)
  NAS_DELAY: 19.5 min avg (28.3%)
  SECURITY_DELAY: 0.1 min avg (0.2%)
  LATE_AIRCRAFT_DELAY: 14.3 min avg (20.8%)

Profile of unpredictable delayed flights:
  Avg DEP_HOUR: 13.6
  Avg DISTANCE: 930
  Avg origin_weather_severity: 0.1794
  Avg airline_delay_rate_30d: 0.2371


In [ ]:
print(f"CRS_ELAPSED_TIME stats:")
print(flights['CRS_ELAPSED_TIME'].describe().round(2))

print(f"\nCorrelation with DISTANCE: {flights['CRS_ELAPSED_TIME'].corr(flights['DISTANCE']):.4f}")

print(f"\nDelay rate by CRS_ELAPSED_TIME bucket:")
flights['etime_bucket'] = pd.cut(flights['CRS_ELAPSED_TIME'], bins=[0, 60, 120, 180, 240, 360, 900])
print(flights.groupby('etime_bucket', observed=True)['ARR_DEL15'].mean().round(4) * 100)
flights.drop(columns=['etime_bucket'], inplace=True)

CRS_ELAPSED_TIME stats:
count    18227796.00
mean          147.06
std            72.92
min            -5.00
25%            93.00
50%           130.00
75%           178.00
max          1162.00
Name: CRS_ELAPSED_TIME, dtype: float64

Correlation with DISTANCE: 0.9832

Delay rate by CRS_ELAPSED_TIME bucket:
etime_bucket
(0, 60]       17.89
(60, 120]     20.21
(120, 180]    22.20
(180, 240]    22.88
(240, 360]    21.55
(360, 900]    21.81
Name: ARR_DEL15, dtype: float64


In [ ]:
weather_cols = {
    'origin_temp': 'Temperature (°C)',
    'origin_dew_point': 'Dew Point',
    'origin_pressure': 'Pressure',
    'origin_wind_dir': 'Wind Direction',
    'origin_wind_speed': 'Wind Speed (m/s)',
    'origin_sky_condition': 'Sky Cover (0-9)',
    'origin_precip_1hr': 'Precipitation (mm)',
}

print("=== WEATHER FEATURE ANALYSIS ===\n")
for col, desc in weather_cols.items():
    print(f"--- {col} ({desc}) ---")
    print(f"  Nulls: {flights[col].isna().sum():,} ({flights[col].isna().mean()*100:.1f}%)")
    print(f"  Stats: min={flights[col].min()}, max={flights[col].max()}, mean={flights[col].mean():.2f}")
    
    non_null = flights[flights[col].notna()]
    quintiles = pd.qcut(non_null[col], q=5, duplicates='drop')
    delay_by_q = non_null.groupby(quintiles, observed=True)['ARR_DEL15'].mean() * 100
    print(f"  Delay by quintile:\n{delay_by_q.round(2).to_string()}")
    print()

=== WEATHER FEATURE ANALYSIS ===

--- origin_temp (Temperature (°C)) ---
  Nulls: 1,570,223 (8.6%)
  Stats: min=-41.7, max=999.9, mean=16.37
  Delay by quintile:
origin_temp
(-41.701, 7.2]    19.34
(7.2, 13.9]       17.49
(13.9, 20.0]      18.86
(20.0, 25.6]      21.71
(25.6, 999.9]     30.62

--- origin_dew_point (Dew Point) ---
  Nulls: 1,570,691 (8.6%)
  Stats: min=-38.3, max=999.9, mean=9.34
  Delay by quintile:
origin_dew_point
(-38.300999999999995, 0.0]    18.86
(0.0, 7.2]                    18.09
(7.2, 12.8]                   20.02
(12.8, 18.9]                  22.22
(18.9, 999.9]                 28.76

--- origin_pressure (Pressure) ---
  Nulls: 1,796,836 (9.9%)
  Stats: min=0.0, max=1049.3, mean=1015.16
  Delay by quintile:
origin_pressure
(-0.001, 1011.0]    24.79
(1011.0, 1014.4]    23.05
(1014.4, 1017.3]    21.80
(1017.3, 1020.9]    20.01
(1020.9, 1049.3]    17.18

--- origin_wind_dir (Wind Direction) ---
  Nulls: 2,073,420 (11.4%)
  Stats: min=0.0, max=999.0, mean=175.56
 

## Weather Sentinel Value Cleaning
NOAA ISD-Lite uses 999.9, 99.9, 999.0 as missing value codes. Replace with NaN.

In [ ]:
sentinel_fixes = {
    'origin_temp': 999.9,
    'origin_dew_point': 999.9,
    'origin_wind_dir': 999.0,
    'origin_wind_speed': 99.9,
    'origin_precip_1hr': 99.9,
    'dest_temp': 999.9,
    'dest_dew_point': 999.9,
    'dest_wind_dir': 999.0,
    'dest_wind_speed': 99.9,
    'dest_precip_1hr': 99.9,
}

for col, sentinel in sentinel_fixes.items():
    count_before = (flights[col] == sentinel).sum()
    flights.loc[flights[col] == sentinel, col] = np.nan
    print(f"{col}: replaced {count_before:,} sentinel values ({sentinel})")

bad_pressure_origin = (flights['origin_pressure'] == 0).sum()
bad_pressure_dest = (flights['dest_pressure'] == 0).sum()
flights.loc[flights['origin_pressure'] == 0, 'origin_pressure'] = np.nan
flights.loc[flights['dest_pressure'] == 0, 'dest_pressure'] = np.nan
print(f"\norigin_pressure: replaced {bad_pressure_origin:,} zero values")
print(f"dest_pressure: replaced {bad_pressure_dest:,} zero values")

print(f"\nAfter fix:")
for col in ['origin_temp', 'origin_dew_point', 'origin_wind_dir', 
            'origin_wind_speed', 'origin_precip_1hr', 'origin_pressure']:
    print(f"  {col}: min={flights[col].min():.1f}, max={flights[col].max():.1f}, nulls={flights[col].isna().sum():,}")

origin_temp: replaced 67 sentinel values (999.9)
origin_dew_point: replaced 925 sentinel values (999.9)
origin_wind_dir: replaced 130,606 sentinel values (999.0)
origin_wind_speed: replaced 2,638 sentinel values (99.9)
origin_precip_1hr: replaced 5,481 sentinel values (99.9)
dest_temp: replaced 86 sentinel values (999.9)
dest_dew_point: replaced 1,033 sentinel values (999.9)
dest_wind_dir: replaced 147,805 sentinel values (999.0)
dest_wind_speed: replaced 3,117 sentinel values (99.9)
dest_precip_1hr: replaced 5,627 sentinel values (99.9)

origin_pressure: replaced 32 zero values
dest_pressure: replaced 33 zero values

After fix:
  origin_temp: min=-41.7, max=99.9, nulls=1,570,290
  origin_dew_point: min=-38.3, max=99.9, nulls=1,571,616
  origin_wind_dir: min=0.0, max=360.0, nulls=2,204,026
  origin_wind_speed: min=0.0, max=34.0, nulls=1,574,817
  origin_precip_1hr: min=-0.1, max=97.7, nulls=2,005,178
  origin_pressure: min=0.1, max=1049.3, nulls=1,796,868


In [ ]:
print("origin_temp > 60°C (140°F):")
print(f"  Count: {(flights['origin_temp'] > 60).sum()}")
print(f"  Values: {sorted(flights[flights['origin_temp'] > 60]['origin_temp'].unique())}")

print("\norigin_dew_point > 40°C:")
print(f"  Count: {(flights['origin_dew_point'] > 40).sum()}")
print(f"  Values: {sorted(flights[flights['origin_dew_point'] > 40]['origin_dew_point'].unique())}")

print("\norigin_precip_1hr > 50mm:")
print(f"  Count: {(flights['origin_precip_1hr'] > 50).sum()}")

print("\norigin_pressure < 900:")
print(f"  Count: {(flights['origin_pressure'] < 900).sum()}")
print(f"  Values: {sorted(flights[flights['origin_pressure'] < 900]['origin_pressure'].unique())[:10]}")

origin_temp > 60°C (140°F):
  Count: 9226
  Values: [np.float64(99.9)]

origin_dew_point > 40°C:
  Count: 10744
  Values: [np.float64(99.9)]

origin_precip_1hr > 50mm:
  Count: 175

origin_pressure < 900:
  Count: 13409
  Values: [np.float64(0.1), np.float64(0.2), np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(0.7), np.float64(0.8), np.float64(0.9), np.float64(1.0)]


In [ ]:
for col in ['origin_temp', 'dest_temp']:
    count = (flights[col] == 99.9).sum()
    flights.loc[flights[col] == 99.9, col] = np.nan
    print(f"{col}: replaced {count:,} values of 99.9")

for col in ['origin_dew_point', 'dest_dew_point']:
    count = (flights[col] == 99.9).sum()
    flights.loc[flights[col] == 99.9, col] = np.nan
    print(f"{col}: replaced {count:,} values of 99.9")

for col in ['origin_pressure', 'dest_pressure']:
    count = (flights[col] < 900).sum()
    flights.loc[flights[col] < 900, col] = np.nan
    print(f"{col}: replaced {count:,} values < 900")

print(f"\nAfter fix:")
for col in ['origin_temp', 'origin_dew_point', 'origin_pressure',
            'origin_wind_speed', 'origin_precip_1hr']:
    print(f"  {col}: min={flights[col].min():.1f}, max={flights[col].max():.1f}")

origin_temp: replaced 9,226 values of 99.9
dest_temp: replaced 9,166 values of 99.9
origin_dew_point: replaced 10,744 values of 99.9
dest_dew_point: replaced 10,600 values of 99.9
origin_pressure: replaced 13,409 values < 900
dest_pressure: replaced 13,364 values < 900

After fix:
  origin_temp: min=-41.7, max=48.3
  origin_dew_point: min=-38.3, max=35.0
  origin_pressure: min=960.3, max=1049.3
  origin_wind_speed: min=0.0, max=34.0
  origin_precip_1hr: min=-0.1, max=97.7


In [ ]:
wind_norm_origin = (flights['origin_wind_speed'] / 30).clip(0, 1)
precip_norm_origin = (flights['origin_precip_1hr'] / 5).clip(0, 1)
sky_norm_origin = flights['origin_sky_condition'] / 9

flights['origin_weather_severity'] = (
    wind_norm_origin * 0.4 + precip_norm_origin * 0.3 + sky_norm_origin * 0.3
).astype('float32')

wind_norm_dest = (flights['dest_wind_speed'] / 30).clip(0, 1)
precip_norm_dest = (flights['dest_precip_1hr'] / 5).clip(0, 1)
sky_norm_dest = flights['dest_sky_condition'] / 9

flights['dest_weather_severity'] = (
    wind_norm_dest * 0.4 + precip_norm_dest * 0.3 + sky_norm_dest * 0.3
).astype('float32')

print("Recomputed weather severity:")
print(f"  origin: {flights['origin_weather_severity'].describe().round(4).to_string()}")
print(f"\n  dest: {flights['dest_weather_severity'].describe().round(4).to_string()}")

flights.to_parquet("dataset/merged_flights_fe.parquet", 
                   index=False, engine="pyarrow", compression="snappy")
import os
size_mb = os.path.getsize("dataset/merged_flights_fe.parquet") / 1024 / 1024
print(f"\nSaved. Size: {size_mb:.1f} MB | Shape: {flights.shape}")

Recomputed weather severity:
  origin: count    9.411395e+06
mean     1.480000e-01
std      1.028000e-01
min      0.000000e+00
25%      6.670000e-02
50%      1.333000e-01
75%      2.200000e-01
max      7.453000e-01

  dest: count    9.184754e+06
mean     1.517000e-01
std      1.004000e-01
min      0.000000e+00
25%      6.800000e-02
50%      1.347000e-01
75%      2.200000e-01
max      7.320000e-01

Saved. Size: 1078.9 MB | Shape: (18227796, 102)


In [ ]:
fin_cols = ['OP_REVENUES', 'OP_EXPENSES', 'NET_INCOME', 'OP_PROFIT_LOSS',
            'FLYING_OPS', 'MAINTENANCE', 'PAX_SERVICE', 'TRANS_REV_PAX',
            'DEPREC_AMORT', 'TRANS_EXPENSES']

print("Correlation with profit_margin:")
for col in fin_cols:
    corr = flights[col].corr(flights['profit_margin'])
    print(f"  {col}: {corr:.4f}")

print("\nCorrelation with ARR_DEL15 (target):")
for col in fin_cols:
    corr = flights[col].corr(flights['ARR_DEL15'])
    print(f"  {col}: {corr:.4f}")

print(f"\nprofit_margin corr with ARR_DEL15: {flights['profit_margin'].corr(flights['ARR_DEL15']):.4f}")

Correlation with profit_margin:
  OP_REVENUES: 0.4212
  OP_EXPENSES: 0.3838
  NET_INCOME: 0.5278
  OP_PROFIT_LOSS: 0.5590
  FLYING_OPS: 0.3261
  MAINTENANCE: 0.3909
  PAX_SERVICE: 0.3794
  TRANS_REV_PAX: 0.4077
  DEPREC_AMORT: 0.3455
  TRANS_EXPENSES: 0.3755

Correlation with ARR_DEL15 (target):
  OP_REVENUES: 0.0007
  OP_EXPENSES: 0.0016
  NET_INCOME: -0.0162
  OP_PROFIT_LOSS: -0.0061
  FLYING_OPS: 0.0068
  MAINTENANCE: 0.0066
  PAX_SERVICE: -0.0033
  TRANS_REV_PAX: 0.0026
  DEPREC_AMORT: -0.0080
  TRANS_EXPENSES: 0.0007

profit_margin corr with ARR_DEL15: 0.0006


In [ ]:
check_features = {
    'is_early_departure': 'DROP candidate (redundant with DEP_HOUR)',
    'is_peak_hours': 'DROP candidate (redundant with DEP_HOUR)', 
    'is_high_delay_day': 'KEEP candidate',
    'is_low_delay_day': 'KEEP candidate',
    'is_summer_peak': 'KEEP candidate',
    'is_fall_low': 'KEEP candidate',
    'maintenance_ratio': 'DROP candidate (mixed signal)',
    'revenue_per_pax': 'DROP candidate (weak signal)',
    'cost_efficiency': 'DROP candidate (redundant with profit_margin)',
    'is_regional': 'KEEP candidate',
    'is_profitable': 'DROP candidate (no signal)',
}

print(f"{'Feature':<25} {'DelayRate=1':<15} {'DelayRate=0':<15} {'Diff':<10} {'Decision'}")
print("=" * 75)

for feat, decision in check_features.items():
    rate_1 = flights[flights[feat] == 1]['ARR_DEL15'].mean() * 100
    rate_0 = flights[flights[feat] == 0]['ARR_DEL15'].mean() * 100
    diff = abs(rate_1 - rate_0)
    print(f"{feat:<25} {rate_1:<15.2f} {rate_0:<15.2f} {diff:<10.2f} {decision}")

Feature                   DelayRate=1     DelayRate=0     Diff       Decision
is_early_departure        12.44           24.91           12.46      DROP candidate (redundant with DEP_HOUR)
is_peak_hours             28.79           16.34           12.45      DROP candidate (redundant with DEP_HOUR)
is_high_delay_day         23.03           19.84           3.18       KEEP candidate
is_low_delay_day          18.71           22.23           3.51       KEEP candidate
is_summer_peak            27.90           19.63           8.27       KEEP candidate
is_fall_low               14.26           22.29           8.02       KEEP candidate
maintenance_ratio         nan             nan             nan        DROP candidate (mixed signal)
revenue_per_pax           18.13           nan             nan        DROP candidate (weak signal)
cost_efficiency           nan             nan             nan        DROP candidate (redundant with profit_margin)
is_regional               18.49           22.14       

In [ ]:
regional_carriers = ['9E', 'MQ', 'OH', 'YX', 'OO']
flights['is_regional_check'] = flights['OP_UNIQUE_CARRIER'].isin(regional_carriers).astype(int)
print(f"is_regional matches carrier-derived flag: {(flights['is_regional'] == flights['is_regional_check']).all()}")
flights.drop(columns=['is_regional_check'], inplace=True)

is_regional matches carrier-derived flag: True


In [ ]:
keep_features = [
    # Time
    'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'ARR_HOUR', 'IS_HOLIDAY', 
    # Flight
    'DISTANCE',
    # Financial
    'profit_margin',
    # Airport
    'origin_monthly_passengers', 'origin_monthly_departures',
    # Weather origin
    'origin_temp', 'origin_dew_point', 'origin_pressure',
    'origin_wind_dir', 'origin_wind_speed', 'origin_sky_condition', 'origin_precip_1hr',
    'origin_weather_severity',
    # Weather dest
    'dest_temp', 'dest_dew_point', 'dest_pressure',
    'dest_wind_dir', 'dest_wind_speed', 'dest_sky_condition', 'dest_precip_1hr',
    'dest_weather_severity',
    # Rolling
    'airline_delay_rate_30d', 'origin_delay_rate_30d', 'dest_delay_rate_30d',
    'route_delay_rate_30d', 'origin_avg_taxi_out_30d',
    'flight_num_delay_rate_30d', 'origin_hour_delay_rate_30d',
    # Cascade
    'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay',
    # Congestion
    'hourly_flight_count',
]

corr = flights[keep_features].corr().round(3)

print("=== HIGH CORRELATION PAIRS (> 0.7) ===\n")
pairs_found = False
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.7:
            pairs_found = True
            print(f"  {corr.columns[i]} <-> {corr.columns[j]}: {corr.iloc[i, j]:.3f}")

if not pairs_found:
    print("  None found!")
keep_features = [
    # Time
    'MONTH', 'DAY_OF_WEEK', 'DEP_HOUR', 'ARR_HOUR', 'IS_HOLIDAY', 
    # Flight
    'DISTANCE',
    # Financial
    'profit_margin',
    # Airport
    'origin_monthly_passengers', 'origin_monthly_departures',
    # Weather origin
    'origin_temp', 'origin_dew_point', 'origin_pressure',
    'origin_wind_dir', 'origin_wind_speed', 'origin_sky_condition', 'origin_precip_1hr',
    'origin_weather_severity',
    # Weather dest
    'dest_temp', 'dest_dew_point', 'dest_pressure',
    'dest_wind_dir', 'dest_wind_speed', 'dest_sky_condition', 'dest_precip_1hr',
    'dest_weather_severity',
    # Rolling
    'airline_delay_rate_30d', 'origin_delay_rate_30d', 'dest_delay_rate_30d',
    'route_delay_rate_30d', 'origin_avg_taxi_out_30d',
    'flight_num_delay_rate_30d', 'origin_hour_delay_rate_30d',
    # Cascade
    'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay',
    # Congestion
    'hourly_flight_count',
]

corr = flights[keep_features].corr().round(3)

print("=== HIGH CORRELATION PAIRS (> 0.7) ===\n")
pairs_found = False
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.7:
            pairs_found = True
            print(f"  {corr.columns[i]} <-> {corr.columns[j]}: {corr.iloc[i, j]:.3f}")

if not pairs_found:
    print("  None found!")

=== HIGH CORRELATION PAIRS (> 0.7) ===

  origin_monthly_passengers <-> origin_monthly_departures: 0.928
  origin_monthly_passengers <-> hourly_flight_count: 0.842
  origin_monthly_departures <-> hourly_flight_count: 0.834
  origin_sky_condition <-> origin_weather_severity: 0.936
  dest_sky_condition <-> dest_weather_severity: 0.932
  origin_delay_rate_30d <-> route_delay_rate_30d: 0.707
=== HIGH CORRELATION PAIRS (> 0.7) ===

  origin_monthly_passengers <-> origin_monthly_departures: 0.928
  origin_monthly_passengers <-> hourly_flight_count: 0.842
  origin_monthly_departures <-> hourly_flight_count: 0.834
  origin_sky_condition <-> origin_weather_severity: 0.936
  dest_sky_condition <-> dest_weather_severity: 0.932
  origin_delay_rate_30d <-> route_delay_rate_30d: 0.707


In [ ]:
print("=== cascade_score distribution ===")
print(flights['cascade_score'].value_counts().sort_index())

print("\n=== Delay rate by cascade_score ===")
print(flights.groupby('cascade_score')['ARR_DEL15'].mean().round(4) * 100)

print("\n=== cascade_delay_minutes stats ===")
print(flights['cascade_delay_minutes'].describe().round(2))

print("\n=== hours_since_last_delay stats ===")
print(flights['hours_since_last_delay'].describe().round(2))
print(f"Nulls: {flights['hours_since_last_delay'].isna().sum():,} ({flights['hours_since_last_delay'].isna().mean()*100:.1f}%)")

print("\n=== N476HA Jan 12 2024 — known cascade day ===")
verify = flights[
    (flights['TAIL_NUM'] == 'N476HA') & 
    (flights['FL_DATE'] == '2024-01-12')
][['CRS_DEP_TIME', 'ORIGIN', 'DEST', 'ARR_DEL15', 'ARR_DELAY',
   'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']
].sort_values('CRS_DEP_TIME')
print(verify.to_string())

print("\n=== N476HA Jan 13 2024 — should be reset ===")
verify2 = flights[
    (flights['TAIL_NUM'] == 'N476HA') & 
    (flights['FL_DATE'] == '2024-01-13')
][['CRS_DEP_TIME', 'ORIGIN', 'DEST', 'ARR_DEL15', 'ARR_DELAY',
   'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']
].sort_values('CRS_DEP_TIME').head(3)
print(verify2.to_string())

print("\n=== N488HA Jan 31 2025 ===")
verify3 = flights[
    (flights['TAIL_NUM'] == 'N488HA') & 
    (flights['FL_DATE'] == '2025-01-31')
][['CRS_DEP_TIME', 'ORIGIN', 'DEST', 'ARR_DEL15', 'ARR_DELAY',
   'cascade_score', 'cascade_delay_minutes', 'hours_since_last_delay']
].sort_values('CRS_DEP_TIME')
print(verify3.to_string())

=== cascade_score distribution ===
cascade_score
0    15649459
1     2053748
2      481206
3       39437
4        3880
5          65
6           1
Name: count, dtype: int64

=== Delay rate by cascade_score ===
cascade_score
0     14.50
1     58.97
2     74.44
3     81.70
4     84.87
5     96.92
6    100.00
Name: ARR_DEL15, dtype: float64

=== cascade_delay_minutes stats ===
count    18227796.00
mean           10.38
std            45.36
min             0.00
25%             0.00
50%             0.00
75%             0.00
max          5426.00
Name: cascade_delay_minutes, dtype: float64

=== hours_since_last_delay stats ===
count    2578337.00
mean           3.24
std            1.22
min            0.00
25%            2.30
50%            3.08
75%            4.10
max            6.00
Name: hours_since_last_delay, dtype: float64
Nulls: 15,649,459 (85.9%)

=== N476HA Jan 12 2024 — known cascade day ===
          CRS_DEP_TIME ORIGIN DEST  ARR_DEL15  ARR_DELAY  cascade_score  cascade_delay_minutes

## Scheduled Turnaround Buffer
Time between previous aircraft's scheduled arrival and current departure. Flights with tight buffers are more vulnerable to cascade delays.

In [ ]:
flights_sorted = flights[['TAIL_NUM', 'FL_DATE', 'CRS_DEP_TIME', 'CRS_ARR_TIME', 'DEP_DATETIME']].copy()
flights_sorted = flights_sorted.sort_values(['TAIL_NUM', 'DEP_DATETIME'])

flights_sorted['prev_crs_arr'] = flights_sorted.groupby('TAIL_NUM')['CRS_ARR_TIME'].shift(1)
flights_sorted['prev_fl_date'] = flights_sorted.groupby('TAIL_NUM')['FL_DATE'].shift(1)

mask = flights_sorted['prev_crs_arr'].notna()

hours = (flights_sorted.loc[mask, 'prev_crs_arr'] // 100).astype(int)
mins = (flights_sorted.loc[mask, 'prev_crs_arr'] % 100).astype(int)

flights_sorted.loc[mask, 'prev_arr_datetime'] = pd.to_datetime(
    flights_sorted.loc[mask, 'prev_fl_date']
) + pd.to_timedelta(hours * 60 + mins, unit='m')

flights_sorted['scheduled_turnaround_buffer'] = (
    (flights_sorted['DEP_DATETIME'] - flights_sorted['prev_arr_datetime']).dt.total_seconds() / 60
).astype('float32')

flights_sorted.loc[
    (flights_sorted['scheduled_turnaround_buffer'] < 0) | 
    (flights_sorted['scheduled_turnaround_buffer'] > 360), 
    'scheduled_turnaround_buffer'
] = np.nan

flights['scheduled_turnaround_buffer'] = flights_sorted['scheduled_turnaround_buffer']

print("=== scheduled_turnaround_buffer ===")
print(f"Nulls: {flights['scheduled_turnaround_buffer'].isna().sum():,} ({flights['scheduled_turnaround_buffer'].isna().mean()*100:.1f}%)")
print(flights['scheduled_turnaround_buffer'].describe().round(2))

print("\n=== N476HA Jan 12 2024 ===")
print(flights[
    (flights['TAIL_NUM'] == 'N476HA') & 
    (flights['FL_DATE'] == '2024-01-12')
][['CRS_DEP_TIME', 'CRS_ARR_TIME', 'ORIGIN', 'DEST', 
   'scheduled_turnaround_buffer']].sort_values('CRS_DEP_TIME').to_string())

=== scheduled_turnaround_buffer ===
Nulls: 4,937,387 (27.1%)
count    13290409.00
mean           66.89
std            44.69
min             0.00
25%            45.00
50%            55.00
75%            70.00
max           360.00
Name: scheduled_turnaround_buffer, dtype: float64

=== N476HA Jan 12 2024 ===
          CRS_DEP_TIME  CRS_ARR_TIME ORIGIN DEST  scheduled_turnaround_buffer
13383190           536           615    HNL  LIH                          NaN
13383192           645           721    LIH  HNL                         30.0
13383194           800           847    HNL  KOA                         39.0
13383193           917          1013    KOA  LIH                         30.0
13383195          1041          1120    LIH  HNL                         28.0
13383197          1209          1252    HNL  LIH                         49.0
13383196          1320          1401    LIH  HNL                         28.0
13383199          1440          1522    HNL  LIH                     

In [ ]:
print("Delay rate by turnaround buffer:")
buckets = pd.cut(flights['scheduled_turnaround_buffer'], bins=[0, 30, 45, 60, 90, 120, 180, 360])
print(flights.groupby(buckets, observed=True)['ARR_DEL15'].mean().round(4) * 100)

Delay rate by turnaround buffer:
scheduled_turnaround_buffer
(0, 30]       36.21
(30, 45]      23.76
(45, 60]      22.38
(60, 90]      19.63
(90, 120]     18.15
(120, 180]    19.71
(180, 360]    22.65
Name: ARR_DEL15, dtype: float64


## Carrier-Origin Rolling Delay Rate (30d)
Captures carrier-specific performance at each origin airport.

In [ ]:
flights['pressure_diff'] = (
    flights['origin_pressure'] - flights['dest_pressure']
).astype('float32')

print("=== pressure_diff ===")
print(f"Nulls: {flights['pressure_diff'].isna().sum():,}")
print(flights['pressure_diff'].describe().round(2))

buckets = pd.cut(flights['pressure_diff'].dropna(), bins=[-50, -10, -5, -2, 2, 5, 10, 50])
print("\nDelay rate by pressure diff:")
print(flights.groupby(buckets, observed=True)['ARR_DEL15'].mean().round(4) * 100)

carrier_origin_daily = (
    flights.groupby(['OP_UNIQUE_CARRIER', 'ORIGIN', 'FL_DATE'])['ARR_DEL15']
    .mean()
    .reset_index()
    .rename(columns={'ARR_DEL15': 'daily_delay_rate'})
)

carrier_origin_daily = carrier_origin_daily.sort_values(['OP_UNIQUE_CARRIER', 'ORIGIN', 'FL_DATE'])

carrier_origin_daily['carrier_origin_delay_rate_30d'] = (
    carrier_origin_daily
    .groupby(['OP_UNIQUE_CARRIER', 'ORIGIN'])['daily_delay_rate']
    .transform(lambda x: x.shift(1).rolling(window=30, min_periods=7).mean())
)

print(f"\n=== carrier_origin_delay_rate_30d ===")
print(f"Shape: {carrier_origin_daily.shape}")
print(f"Nulls: {carrier_origin_daily['carrier_origin_delay_rate_30d'].isna().sum():,}")

flights = flights.merge(
    carrier_origin_daily[['OP_UNIQUE_CARRIER', 'ORIGIN', 'FL_DATE', 'carrier_origin_delay_rate_30d']],
    on=['OP_UNIQUE_CARRIER', 'ORIGIN', 'FL_DATE'],
    how='left'
)

print(f"\nShape after merge: {flights.shape}")
print(f"Nulls in flights: {flights['carrier_origin_delay_rate_30d'].isna().sum():,} ({flights['carrier_origin_delay_rate_30d'].isna().mean()*100:.1f}%)")

=== pressure_diff ===
Nulls: 3,422,708
count    14805088.00
mean            0.08
std             6.88
min           -61.90
25%            -3.70
50%             0.10
75%             3.90
max            62.40
Name: pressure_diff, dtype: float64

Delay rate by pressure diff:
pressure_diff
(-50, -10]    22.08
(-10, -5]     21.51
(-5, -2]      21.25
(-2, 2]       20.99
(2, 5]        21.58
(5, 10]       21.64
(10, 50]      21.99
Name: ARR_DEL15, dtype: float64

=== carrier_origin_delay_rate_30d ===
Shape: (1352926, 5)
Nulls: 12,762

Shape after merge: (18227796, 106)
Nulls in flights: 119,862 (0.7%)


In [ ]:
flights.drop(columns=['pressure_diff'], inplace=True)
print(f"Dropped pressure_diff. Shape: {flights.shape}")

Dropped pressure_diff. Shape: (18227796, 105)


In [ ]:
print("Delay rate by carrier_origin_delay_rate_30d quintile:")
non_null = flights[flights['carrier_origin_delay_rate_30d'].notna()]
quintiles = pd.qcut(non_null['carrier_origin_delay_rate_30d'], q=5, duplicates='drop')
print(non_null.groupby(quintiles, observed=True)['ARR_DEL15'].mean().round(4) * 100)

print(f"\nCorrelation with existing rolling features:")
print(flights[['carrier_origin_delay_rate_30d', 'airline_delay_rate_30d', 
               'origin_delay_rate_30d', 'route_delay_rate_30d']].corr().round(4))

Delay rate by carrier_origin_delay_rate_30d quintile:
carrier_origin_delay_rate_30d
(-0.001, 0.137]    12.66
(0.137, 0.181]     17.17
(0.181, 0.224]     20.55
(0.224, 0.283]     24.14
(0.283, 1.0]       31.54
Name: ARR_DEL15, dtype: float64

Correlation with existing rolling features:
                               carrier_origin_delay_rate_30d  \
carrier_origin_delay_rate_30d                         1.0000   
airline_delay_rate_30d                                0.7497   
origin_delay_rate_30d                                 0.7935   
route_delay_rate_30d                                  0.7314   

                               airline_delay_rate_30d  origin_delay_rate_30d  \
carrier_origin_delay_rate_30d                  0.7497                 0.7935   
airline_delay_rate_30d                         1.0000                 0.6417   
origin_delay_rate_30d                          0.6417                 1.0000   
route_delay_rate_30d                           0.5659                 0.7

## Save Dataset V2

In [ ]:
flights.to_parquet("dataset/merged_flights_fe.parquet", 
                   index=False, engine="pyarrow", compression="snappy")
import os
size_mb = os.path.getsize("dataset/merged_flights_fe.parquet") / 1024 / 1024
print(f"Saved. Size: {size_mb:.1f} MB | Shape: {flights.shape}")


Saved. Size: 1151.6 MB | Shape: (18227796, 105)
